In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

import os

In [ ]:
from pathlib import Path
import pandas as pd

# Kaggle competition data is intentionally not redistributed in this repository.
# Put your downloaded train.csv and test.csv in a local data/ directory.
DATA_DIR_CANDIDATES = [Path.cwd() / "data", Path.cwd(), Path.cwd().parent / "data", Path.cwd().parent]
DATA_DIR = next(
    (directory for directory in DATA_DIR_CANDIDATES
     if (directory / "train.csv").is_file() and (directory / "test.csv").is_file()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "train.csv and test.csv were not found. Download the files from the "
        "appropriate Kaggle competition and place both in a data/ folder."
    )

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
id = test["id"]
print(f"Loaded competition data from: {DATA_DIR.resolve()}")

In [ ]:
train

In [ ]:
train.describe()

In [ ]:
cases = [train,test]

for case in cases:
  print(case.describe())
  print('/n')

In [ ]:
train_coloumn=['Episode_Length_minutes','Guest_Popularity_percentage','Number_of_Ads']
for col in train_coloumn:
   train[col]=train[col].fillna(train[col].median())
train.info()

test_coloumn=['Episode_Length_minutes','Guest_Popularity_percentage']
for col in test_coloumn:
   test[col]=test[col].fillna(test[col].median())
test.info()

In [ ]:

cols=['id','Podcast_Name']
for col in cols:
  train=train.drop(col,axis=1)
  test=test.drop(col,axis=1)

In [ ]:
cols = ['Episode_Length_minutes', 'Number_of_Ads']
for col in cols: # Changed 'clos' to 'cols'
 plt.figure()
 sns.boxplot(x=train[col]) # Changed 'traid' to 'train'
 plt.show()

In [ ]:
train_coloumn=['Episode_Length_minutes','Number_of_Ads']
for col in train_coloumn:
    low,high=train[col].quantile([0.01,0.99])
    train[col]=train[col].clip(lower=low,upper=high)
train.describe()

In [ ]:
test_coloumn=['Episode_Length_minutes','Number_of_Ads']
for col in test_coloumn:
    low,high=test[col].quantile([0.01,0.99])
    train[col]=test[col].clip(lower=low,upper=high)
test.describe()

In [ ]:
from sklearn.preprocessing import LabelEncoder
cattrain=['Episode_Title','Genre','Publication_Day','Publication_Time','Number_of_Ads','Episode_Sentiment']
le=LabelEncoder()
for col in cattrain:
        train[col]=le.fit_transform(train[col])
        test[col]=le.fit_transform(test[col])
train.head()
test.head()

In [ ]:
#Making both the guest and host popularity as combine rating out of 10
train['Rating_GH']=(((train['Host_Popularity_percentage']/100+train['Guest_Popularity_percentage']/100)/2)*10).round(0).astype(int)
test['Rating_GH']=(((test['Host_Popularity_percentage']/100+test['Guest_Popularity_percentage']/100)/2)*10).round(0).astype(int)
train.head()

In [ ]:
#Splited the data with the help of train_test_split with the split of 80/20
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

x=train.drop(columns=['Listening_Time_minutes'])
y=train['Listening_Time_minutes']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
from lightgbm import LGBMRegressor
model=LGBMRegressor(random_state=42)
model.fit(x_train,y_train) #trained the data
ypred=model.predict(x_test) #tested the data

rmse=np.sqrt(mean_squared_error(y_test,ypred))
print("rmse:",rmse)

In [ ]:
from xgboost import XGBRegressor
model=XGBRegressor(random_state=42)
model.fit(x_train,y_train) #trained the data
ypred=model.predict(x_test) #tested the data

rmse=np.sqrt(mean_squared_error(y_test,ypred))
print("rmse:",rmse)

In [ ]:
pred=model.predict(test) #predicting with actual testdata
submission=pd.DataFrame({
    'id':id,
    'Listening_Time_minutes': pred
})
submission.to_csv('submission.csv',index=False)

In [ ]:
submission.head()

In [ ]:
from pathlib import Path
print(f"Submission saved to: {(Path.cwd() / 'submission.csv').resolve()}")
print("Upload submission.csv to the matching Kaggle competition after checking its schema.")